# Direct Marketing with Amazon SageMaker XGBoost and Hyperparameter Tuning (SageMaker API)
_**Supervised Learning with Gradient Boosted Trees: A Binary Prediction Problem With Unbalanced Classes**_

---

---

Kernel `Python 3 (Data Science)` works well with this notebook.

## Contents

1. [Background](#Background)
1. [Prepration](#Preparation)
1. [Data Downloading](#Data_Downloading)
1. [Data Transformation](#Data_Transformation)
1. [Setup Hyperparameter Tuning](#Setup_Hyperparameter_Tuning)
1. [Launch Hyperparameter Tuning](#Launch_Hyperparameter_Tuning)
1. [Analyze Hyperparameter Tuning Results](#Analyze_Hyperparameter_Tuning_Results)
1. [Deploy The Best Model](#Deploy_The_Best_Model)


---

## Background
Direct marketing, either through mail, email, phone, etc., is a common tactic to acquire customers.  Because resources and a customer's attention is limited, the goal is to only target the subset of prospects who are likely to engage with a specific offer.  Predicting those potential customers based on readily available information like demographics, past interactions, and environmental factors is a common machine learning problem.

This notebook will train a model which can be used to predict if a customer will enroll for a term deposit at a bank, after one or more phone calls. Hyperparameter tuning will be used in order to try multiple hyperparameter settings and produce the best model.

---

## Preparation

Let's start by specifying:

- The S3 bucket and prefix that for the training and model data.  This will be within the same region as SageMaker training.
- The IAM role used to give training access to the data.

In [ ]:
# Install necessary libraries
%pip install bokeh

In [1]:
import sagemaker
import boto3

import numpy as np  # For matrix operations and numerical processing
import pandas as pd  # For munging tabular data
from time import gmtime, strftime
import os

region = boto3.Session().region_name
smclient = boto3.Session().client("sagemaker")

role = sagemaker.get_execution_role()

bucket = "aigc-training-1668"
prefix = "module8/sagemaker/DEMO-hpo-xgboost-dm"

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


---

## Data_Downloading
This project uses [direct marketing dataset](https://archive.ics.uci.edu/ml/datasets/bank+marketing) from UCI's ML Repository.

In [2]:
!wget -N https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
!unzip -o bank-additional.zip

--2026-04-02 21:29:30--  https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Last-modified header missing -- time-stamps turned off.
--2026-04-02 21:29:30--  https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
Reusing existing connection to archive.ics.uci.edu:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘bank-additional.zip’

    [   <=>                                 ] 444,572     1.05MB/s   in 0.4s   

2026-04-02 21:29:31 (1.05 MB/s) - ‘bank-additional.zip’ saved [444572]

Archive:  bank-additional.zip
  inflating: bank-additional/.DS_Store  
  inflating: __MACOSX/bank-additional/._.DS_Store  
  inflating: bank-additional/.Rhistory  
  inflating: bank-additio

In [3]:
# Read data into a dataframe and examine it
data = pd.read_csv("./bank-additional/bank-additional-full.csv", sep=";")
pd.set_option("display.max_columns", 500)  # Make sure we can see all of the columns
pd.set_option("display.max_rows", 50)  # Keep the output on one page
data

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,334,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,383,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,189,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,442,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


Overview:

* We have a little over 40K customer records, and 20 features for each customer
* The features are mixed; some numeric, some categorical
* The data appears to be sorted, at least by `time` and `contact`, maybe more

_**Specifics on each of the features:**_

*Demographics:*
* `age`: Customer's age (numeric)
* `job`: Type of job (categorical: 'admin.', 'services', ...)
* `marital`: Marital status (categorical: 'married', 'single', ...)
* `education`: Level of education (categorical: 'basic.4y', 'high.school', ...)

*Past customer events:*
* `default`: Has credit in default? (categorical: 'no', 'unknown', ...)
* `housing`: Has housing loan? (categorical: 'no', 'yes', ...)
* `loan`: Has personal loan? (categorical: 'no', 'yes', ...)

*Past direct marketing contacts:*
* `contact`: Contact communication type (categorical: 'cellular', 'telephone', ...)
* `month`: Last contact month of year (categorical: 'may', 'nov', ...)
* `day_of_week`: Last contact day of the week (categorical: 'mon', 'fri', ...)
* `duration`: Last contact duration, in seconds (numeric). Important note: If duration = 0 then `y` = 'no'.
 
*Campaign information:*
* `campaign`: Number of contacts performed during this campaign and for this client (numeric, includes last contact)
* `pdays`: Number of days that passed by after the client was last contacted from a previous campaign (numeric)
* `previous`: Number of contacts performed before this campaign and for this client (numeric)
* `poutcome`: Outcome of the previous marketing campaign (categorical: 'nonexistent','success', ...)

*External environment factors:*
* `emp.var.rate`: Employment variation rate - quarterly indicator (numeric)
* `cons.price.idx`: Consumer price index - monthly indicator (numeric)
* `cons.conf.idx`: Consumer confidence index - monthly indicator (numeric)
* `euribor3m`: Euribor 3 month rate - daily indicator (numeric)
* `nr.employed`: Number of employees - quarterly indicator (numeric)

*Target variable:*
* `y`: Has the client subscribed a term deposit? (binary: 'yes','no')

## Data_Transformation
Cleaning up data is part of nearly every machine learning project.  It arguably presents the biggest risk if done incorrectly and is one of the more subjective aspects in the process.  Several common techniques include:

* Handling missing values: Some machine learning algorithms are capable of handling missing values, but most would rather not.  Options include:
 * Removing observations with missing values: This works well if only a very small fraction of observations have incomplete information.
 * Removing features with missing values: This works well if there are a small number of features which have a large number of missing values.
 * Imputing missing values: Entire [books](https://www.amazon.com/Flexible-Imputation-Missing-Interdisciplinary-Statistics/dp/1439868247) have been written on this topic, but common choices are replacing the missing value with the mode or mean of that column's non-missing values.
* Converting categorical to numeric: The most common method is one hot encoding, which for each feature maps every distinct value of that column to its own feature which takes a value of 1 when the categorical feature is equal to that value, and 0 otherwise.
* Oddly distributed data: Although for non-linear models like Gradient Boosted Trees, this has very limited implications, parametric models like regression can produce wildly inaccurate estimates when fed highly skewed data.  In some cases, simply taking the natural log of the features is sufficient to produce more normally distributed data.  In others, bucketing values into discrete ranges is helpful.  These buckets can then be treated as categorical variables and included in the model when one hot encoded.
* Handling more complicated data types: Mainpulating images, text, or data at varying grains.

Some of these aspects have already been handled for this dataset, and the algorithms in this project tends to do well at handling sparse or oddly distributed data.

First of all, Many records have the value of "999" for pdays, number of days that passed by after a client was last contacted. It is very likely to be a magic number to represent that no contact was made before. Considering that, we create a new column called "no_previous_contact", then grant it value of "1" when pdays is 999 and "0" otherwise.

In the "job" column, there are categories that mean the customer is not working, e.g., "student", "retire", and "unemployed". Since it is very likely whether or not a customer is working will affect his/her decision to enroll in the term deposit, we generate a new column to show whether the customer is working based on "job" column.

Last but not the least, we convert categorical to numeric, as is suggested above.

In [4]:
data["no_previous_contact"] = np.where(
    data["pdays"] == 999, 1, 0
)  # Indicator variable to capture when pdays takes a value of 999
data["not_working"] = np.where(
    np.in1d(data["job"], ["student", "retired", "unemployed"]), 1, 0
)  # Indicator for individuals not actively employed
model_data = pd.get_dummies(data)  # Convert categorical variables to sets of indicators
model_data

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,no_previous_contact,not_working,job_admin.,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown,marital_divorced,marital_married,marital_single,marital_unknown,education_basic.4y,education_basic.6y,education_basic.9y,education_high.school,education_illiterate,education_professional.course,education_university.degree,education_unknown,default_no,default_unknown,default_yes,housing_no,housing_unknown,housing_yes,loan_no,loan_unknown,loan_yes,contact_cellular,contact_telephone,month_apr,month_aug,month_dec,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,day_of_week_fri,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_failure,poutcome_nonexistent,poutcome_success,y_no,y_yes
0,56,261,1,999,0,1.1,93.994,-36.4,4.857,5191.0,1,0,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False
1,57,149,1,999,0,1.1,93.994,-36.4,4.857,5191.0,1,0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False
2,37,226,1,999,0,1.1,93.994,-36.4,4.857,5191.0,1,0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False
3,40,151,1,999,0,1.1,93.994,-36.4,4.857,5191.0,1,0,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False
4,56,307,1,999,0,1.1,93.994,-36.4,4.857,5191.0,1,0,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,False,True,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,334,1,999,0,-1.1,94.767,-50.8,1.028,4963.6,1,1,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True
41184,46,383,1,999,0,-1.1,94.767,-50.8,1.028,4963.6,1,0,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,True,False
41185,56,189,2,999,0,-1.1,94.767,-50.8,1.028,4963.6,1,1,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,Fals

Another question to ask before building a model is whether certain features will add value in the final use case.  For example, if the goal is to deliver the best prediction, then will we have access to that data at the moment of prediction?  Knowing it's raining is highly predictive for umbrella sales, but forecasting weather far enough out to plan inventory on umbrellas is probably just as difficult as forecasting umbrella sales without knowledge of the weather.  So, including this in the model may give a false sense of precision.

Following this logic, remove the economic features and `duration` from the data as they would need to be forecasted with high precision to use as inputs in future predictions.

Even if we were to use values of the economic indicators from the previous quarter, this value is likely not as relevant for prospects contacted early in the next quarter as those contacted later on.

In [5]:
model_data = model_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)

This step splits the dataset into training (70%), validation (20%), and test (10%) datasets and convert the datasets to the right format the algorithm expects. We will use training and validation datasets during training. Test dataset will be used to evaluate model performance after it is deployed to an endpoint.

Amazon SageMaker's XGBoost algorithm expects data in the libSVM or CSV data format.  For this example, we'll stick to CSV.  Note that the first column must be the target variable and the CSV should not include headers.  Also, notice that although repetitive it's easiest to do this after the train|validation|test split rather than before.  This avoids any misalignment issues due to random reordering.

In [6]:
train_data, validation_data, test_data = np.split(
    model_data.sample(frac=1, random_state=1729),
    [int(0.7 * len(model_data)), int(0.9 * len(model_data))],
)

pd.concat([train_data["y_yes"], train_data.drop(["y_no", "y_yes"], axis=1)], axis=1).to_csv(
    "train.csv", index=False, header=False
)
pd.concat(
    [validation_data["y_yes"], validation_data.drop(["y_no", "y_yes"], axis=1)], axis=1
).to_csv("validation.csv", index=False, header=False)
pd.concat([test_data["y_yes"], test_data.drop(["y_no", "y_yes"], axis=1)], axis=1).to_csv(
    "test.csv", index=False, header=False
)

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Now we'll copy the file to S3 for Amazon SageMaker training to pickup.

In [5]:
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "train/train.csv")
).upload_file("train.csv")
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "validation/validation.csv")
).upload_file("validation.csv")

---

## Setup_Hyperparameter_Tuning 

This section will use SageMaker hyperparameter tuning to automate the searching process effectively. SageMaker hyperparameter tuning will automatically launch multiple training jobs with different hyperparameter settings, evaluate results of those training jobs based on a predefined "objective metric", and select the hyperparameter settings for future attempts based on previous results.

First configure the hyperparameter tuning job by defining a JSON object that specifies following information:
* The ranges of hyperparameters to tune
* Number of training jobs to run in total and how many training jobs should be run simultaneously.
* The objective metric that will be used to evaluate training results

Tune four hyperparameters in this project:
* *eta*: Step size shrinkage used in updates to prevent overfitting. After each boosting step, directly get the weights of new features. The eta parameter actually shrinks the feature weights to make the boosting process more conservative. 
* *alpha*: L1 regularization term on weights. Increasing this value makes models more conservative. 
* *min_child_weight*: Minimum sum of instance weight (hessian) needed in a child. If the tree partition step results in a leaf node with the sum of instance weight less than min_child_weight, the building process gives up further partitioning. In linear regression models, this simply corresponds to a minimum number of instances needed in each node. The larger the algorithm, the more conservative it is. 
* *max_depth*: Maximum depth of a tree. Increasing this value makes the model more complex and likely to be overfitted. 

In [6]:
from time import gmtime, strftime, sleep

tuning_job_name = "xgboost-tuningjob-" + strftime("%d-%H-%M-%S", gmtime())

print(tuning_job_name)

tuning_job_config = {
    "ParameterRanges": {
        "CategoricalParameterRanges": [],
        "ContinuousParameterRanges": [
            {
                "MaxValue": "1",
                "MinValue": "0",
                "Name": "eta",
            },
            {
                "MaxValue": "10",
                "MinValue": "1",
                "Name": "min_child_weight",
            },
            {
                "MaxValue": "2",
                "MinValue": "0",
                "Name": "alpha",
            },
        ],
        "IntegerParameterRanges": [
            {
                "MaxValue": "10",
                "MinValue": "1",
                "Name": "max_depth",
            }
        ],
    },
    "ResourceLimits": {"MaxNumberOfTrainingJobs": 10, "MaxParallelTrainingJobs": 3},
    "Strategy": "Bayesian",
    "HyperParameterTuningJobObjective": {"MetricName": "validation:mse", "Type": "Minimize"},
}

xgboost-tuningjob-31-13-18-07


Now configure the training jobs the hyperparameter tuning job will launch by defining a JSON object that specifies following information:
* The container image for the algorithm (XGBoost)
* The input configuration for the training and validation data
* Configuration for the output of the algorithm
* The values of any algorithm hyperparameters that are not tuned in the tuning job (StaticHyperparameters)
* The type and number of instances to use for the training jobs
* The stopping condition for the training jobs

In [7]:
from sagemaker.image_uris import retrieve

training_image = retrieve(framework="xgboost", region=region, version="1.5-1")

s3_input_train = "s3://{}/{}/train".format(bucket, prefix)
s3_input_validation = "s3://{}/{}/validation/".format(bucket, prefix)

training_job_definition = {
    "AlgorithmSpecification": {"TrainingImage": training_image, "TrainingInputMode": "File"},
    "InputDataConfig": [
        {
            "ChannelName": "train",
            "CompressionType": "None",
            "ContentType": "csv",
            "DataSource": {
                "S3DataSource": {
                    "S3DataDistributionType": "FullyReplicated",
                    "S3DataType": "S3Prefix",
                    "S3Uri": s3_input_train,
                }
            },
        },
        {
            "ChannelName": "validation",
            "CompressionType": "None",
            "ContentType": "csv",
            "DataSource": {
                "S3DataSource": {
                    "S3DataDistributionType": "FullyReplicated",
                    "S3DataType": "S3Prefix",
                    "S3Uri": s3_input_validation,
                }
            },
        },
    ],
    "OutputDataConfig": {"S3OutputPath": "s3://{}/{}/output".format(bucket, prefix)},
    "ResourceConfig": {"InstanceCount": 1, "InstanceType": "ml.m4.xlarge", "VolumeSizeInGB": 10},
    "RoleArn": role,
    "StaticHyperParameters": {
        "eval_metric": "auc",
        "num_round": "100",
        "objective": "binary:hinge",
        "rate_drop": "0.3",
        "tweedie_variance_power": "1.4",
    },
    "StoppingCondition": {"MaxRuntimeInSeconds": 43200},
}

## Launch_Hyperparameter_Tuning
Now launch a hyperparameter tuning job by calling create_hyper_parameter_tuning_job API.

In [11]:
smclient.create_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name,
    HyperParameterTuningJobConfig=tuning_job_config,
    TrainingJobDefinition=training_job_definition,
)

{'HyperParameterTuningJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:hyper-parameter-tuning-job/xgboost-tuningjob-31-12-47-50',
 'ResponseMetadata': {'RequestId': 'b17bd942-bbf7-4300-ad93-4f96b238afaa',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'b17bd942-bbf7-4300-ad93-4f96b238afaa',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '130',
   'date': 'Tue, 31 Mar 2026 12:50:20 GMT'},
  'RetryAttempts': 0}}

Let's just run a quick check of the hyperparameter tuning jobs status to make sure it started successfully.

In [31]:
smclient.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)[
    "HyperParameterTuningJobStatus"
]

'Completed'

In [32]:
smclient.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=tuning_job_name)['HyperParameterTuningJobName']

'xgboost-tuningjob-31-12-47-50'

### Important: ### 
Waits until the hyperparameter job is done. The progress can also be tracked via the AWS console.

In [33]:
# run this cell to check current status of hyperparameter tuning job
tuning_job_result = smclient.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuning_job_name
)

status = tuning_job_result["HyperParameterTuningJobStatus"]
if status != "Completed":
    print("Reminder: the tuning job has not been completed.")

job_count = tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" % job_count)

objective = tuning_job_result["HyperParameterTuningJobConfig"]["HyperParameterTuningJobObjective"]
is_minimize = objective["Type"] != "Maximize"
objective_name = objective["MetricName"]

10 training jobs have completed


Query the best training job programmatically:

In [34]:
from pprint import pprint

if tuning_job_result.get("BestTrainingJob", None):
    print("Best model found so far:")
    pprint(tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")

Best model found so far:
{'CreationTime': datetime.datetime(2026, 3, 31, 12, 53, 46, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:mse',
                                                 'Value': 1.076550006866455},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 3, 31, 12, 54, 30, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:training-job/xgboost-tuningjob-31-12-47-50-004-1a0cbaf8',
 'TrainingJobName': 'xgboost-tuningjob-31-12-47-50-004-1a0cbaf8',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2026, 3, 31, 12, 53, 51, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '0.14691420509441366',
                          'eta': '0.7688158018995402',
                          'max_depth': '2',
                          'min_child_weight': '1.3942415759088362'}}


## Fetch all results as DataFrame
See all the training jobs and list hyperparameters and objective metrics and pick up the training job with the best objective metric.

In [35]:
import pandas as pd

tuner = sagemaker.HyperparameterTuningJobAnalytics(tuning_job_name)

full_df = tuner.dataframe()

if len(full_df) > 0:
    df = full_df[full_df["FinalObjectiveValue"] > -float("inf")]
    if len(df) > 0:
        df = df.sort_values("FinalObjectiveValue", ascending=is_minimize)
        print("Number of training jobs with valid objective: %d" % len(df))
        print({"lowest": min(df["FinalObjectiveValue"]), "highest": max(df["FinalObjectiveValue"])})
        pd.set_option("display.max_colwidth", None)  # Don't truncate TrainingJobName
    else:
        print("No training jobs have reported valid results yet.")

full_df

Number of training jobs with valid objective: 10
{'lowest': 1.076550006866455, 'highest': 2.183029890060425}


,alpha,eta,max_depth,min_child_weight,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,0.417563,0.515321,2.0,2.364923,xgboost-tuningjob-31-12-47-50-010-1eb7dbd9,Completed,1.09388,2026-03-31 12:56:34+00:00,2026-03-31 12:57:13+00:00,39.0
1,0.389558,0.880884,10.0,2.197813,xgboost-tuningjob-31-12-47-50-009-e2b65b83,Completed,1.59184,2026-03-31 12:56:34+00:00,2026-03-31 12:57:13+00:00,39.0
2,0.434511,0.526219,2.0,1.000000,xgboost-tuningjob-31-12-47-50-008-0a04f912,Completed,1.16333,2026-03-31 12:55:09+00:00,2026-03-31 12:55:48+00:00,39.0
3,0.000000,0.905977,7.0,2.304936,xgboost-tuningjob-31-12-47-50-007-4d21b8fa,Completed,1.72106,2026-03-31 12:54:50+00:00,2026-03-31 12:55:28+00:00,38.0
4,0.000000,0.119850,10.0,1.000000,xgboost-tuningjob-31-12-47-50-006-39370d95,Completed,1.11935,2026-03-31 12:54:18+00:00,2026-03-31 12:55:08+00:00,50.0
5,0.690987,0.659208,2.0,9.231795,xgboost-tuningjob-31-12-47-50-005-c59ce7c7,Completed,2.18303,2026-03-31 12:54:12+00:00,2026-03-31 12:54:51+00:00,39.0
6,0.146914,0.768816,2.0,1.394242,xgboost-tuningjob-31-12-47-50-004-1a0cbaf8,Completed,1.07655,2026-03-31 12:53:51+00:00,2026-03-31 12:54:30+00:00,39.0
7,1.879904,0.550714,9.0,1.065638,xgboost-tuningjob-31-12-47-50-003-4e0fc060,Completed,1.32705,2026-03-31 12:51:18+00:00,2026-03-31 12:53:28+00:00,130.0
8,1.601882,0.359139,5.0,9.178125,xgboost-tuningjob-31-12-47-50-002-eb6753e7,Completed,1.67840,2026-03-31 12:51:25+00:00,2026-03-31 12:53:30+00:00,125.0
9,1.201225,0.909928,1.0,9.354734,xgboost-tuningjob-31-12-47-50-001-dbc842ac,Completed,1.74165,2026-03-31 12:51:34+00:00,2026-03-31 12:53:54+00:00,140.0


## See TuningJob results vs time
Examine how the objective metric changes over time, as the tuning job progresses.

In [36]:
df.head()

,alpha,eta,max_depth,min_child_weight,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
6,0.146914,0.768816,2.0,1.394242,xgboost-tuningjob-31-12-47-50-004-1a0cbaf8,Completed,1.07655,2026-03-31 12:53:51+00:00,2026-03-31 12:54:30+00:00,39.0
0,0.417563,0.515321,2.0,2.364923,xgboost-tuningjob-31-12-47-50-010-1eb7dbd9,Completed,1.09388,2026-03-31 12:56:34+00:00,2026-03-31 12:57:13+00:00,39.0
4,0.000000,0.119850,10.0,1.000000,xgboost-tuningjob-31-12-47-50-006-39370d95,Completed,1.11935,2026-03-31 12:54:18+00:00,2026-03-31 12:55:08+00:00,50.0
2,0.434511,0.526219,2.0,1.000000,xgboost-tuningjob-31-12-47-50-008-0a04f912,Completed,1.16333,2026-03-31 12:55:09+00:00,2026-03-31 12:55:48+00:00,39.0
7,1.879904,0.550714,9.0,1.065638,xgboost-tuningjob-31-12-47-50-003-4e0fc060,Completed,1.32705,2026-03-31 12:51:18+00:00,2026-03-31 12:53:28+00:00,130.0


In [17]:
import bokeh
import bokeh.io

bokeh.io.output_notebook()
from bokeh.plotting import figure, show
from bokeh.models import HoverTool


class HoverHelper:
    def __init__(self, tuning_analytics):
        self.tuner = tuning_analytics

    def hovertool(self):
        tooltips = [
            ("FinalObjectiveValue", "@FinalObjectiveValue"),
            ("TrainingJobName", "@TrainingJobName"),
        ]
        for k in self.tuner.tuning_ranges.keys():
            tooltips.append((k, "@{%s}" % k))

        ht = HoverTool(tooltips=tooltips)
        return ht

    def tools(self, standard_tools="pan,crosshair,wheel_zoom,zoom_in,zoom_out,undo,reset"):
        return [self.hovertool(), standard_tools]

Loading BokehJS ...

In [37]:
hover = HoverHelper(tuner)

p = figure(width=900, height=400, tools=hover.tools(), x_axis_type="datetime")
p.scatter(source=df, x="TrainingStartTime", y="FinalObjectiveValue",  marker="circle", size=10)
show(p)

## Analyze the correlation between objective metric and individual hyperparameters
Examine the correlation between the objective metric and individual hyperparameters we've selected to tune.

In [38]:
ranges = tuner.tuning_ranges
figures = []
for hp_name, hp_range in ranges.items():
    categorical_args = {}
    if hp_range.get("Values"):
        # This is marked as categorical.  Check if all options are actually numbers.
        def is_num(x):
            try:
                float(x)
                return 1
            except:
                return 0

        vals = hp_range["Values"]
        if sum([is_num(x) for x in vals]) == len(vals):
            # Bokeh has issues plotting a "categorical" range that's actually numeric, so plot as numeric
            print("Hyperparameter %s is tuned as categorical, but all values are numeric" % hp_name)
        else:
            # Set up extra options for plotting categoricals.  A bit tricky when they're actually numbers.
            categorical_args["x_range"] = vals

    # Now plot it
    p = figure(
        width=500,
        height=500,
        title="Objective vs %s" % hp_name,
        tools=hover.tools(),
        x_axis_label=hp_name,
        y_axis_label=objective_name,
        **categorical_args,
    )
    p.scatter(source=df, x=hp_name, y="FinalObjectiveValue", marker="circle", size=10)
    figures.append(p)
show(bokeh.layouts.Column(*figures))

# Amazon Sagemaker Hyperparameter Tuning Strategy Explorations

## GridSearch
GridSearch only allows Categorical parameters, so we must convert the continuous and integer parameters into suitable selections. To avoid blowing up the training job count expontentially, we will limit to 2 categories per parameter (16 jobs)

In [7]:
from time import gmtime, strftime, sleep

grid_tuning_job_name = "xg-grid-tuningjob-" + strftime("%d-%H-%M-%S", gmtime())

print(grid_tuning_job_name)

grid_tuning_job_config = {
    "ParameterRanges": {
        "CategoricalParameterRanges": [
            {
                "Name": "eta",
                "Values": ["0.3", "0.7"]
            },
            {
                "Name": "min_child_weight",
                "Values": ["3", "7"]
            },
            {
                "Name": "alpha",
                "Values": ["0.5", "1.5"]
            },
            {
                "Name": "max_depth",
                "Values": ["3", "7"]
            }
        ],
    },
    # Remove MaxNumberOfTrainingJobs since automatically set to all possible combinations
    "ResourceLimits": {"MaxParallelTrainingJobs": 3},
    "Strategy": "Grid",
    "HyperParameterTuningJobObjective": {"MetricName": "validation:mse", "Type": "Minimize"},
}

xg-grid-tuningjob-02-21-30-36


In [38]:
smclient.create_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=grid_tuning_job_name,
    HyperParameterTuningJobConfig=grid_tuning_job_config,
    TrainingJobDefinition=training_job_definition,
)

CPU times: user 11.5 ms, sys: 4.66 ms, total: 16.2 ms
Wall time: 815 ms


{'HyperParameterTuningJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:hyper-parameter-tuning-job/xg-grid-tuningjob-31-14-54-24',
 'ResponseMetadata': {'RequestId': '9342364b-d625-4ea2-89fd-95564b664e93',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '9342364b-d625-4ea2-89fd-95564b664e93',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '130',
   'date': 'Tue, 31 Mar 2026 14:55:29 GMT'},
  'RetryAttempts': 0}}

In [9]:
smclient.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=grid_tuning_job_name)[
    "HyperParameterTuningJobStatus"
]

'Completed'

In [10]:
smclient.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=grid_tuning_job_name)['HyperParameterTuningJobName']

'xg-grid-tuningjob-31-14-54-24'

In [11]:
# run this cell to check current status of hyperparameter tuning job
grid_tuning_job_result = smclient.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=grid_tuning_job_name
)

status = grid_tuning_job_result["HyperParameterTuningJobStatus"]
if status != "Completed":
    print("Reminder: the tuning job has not been completed.")

job_count = grid_tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" % job_count)

objective = grid_tuning_job_result["HyperParameterTuningJobConfig"]["HyperParameterTuningJobObjective"]
is_minimize = objective["Type"] != "Maximize"
objective_name = objective["MetricName"]

16 training jobs have completed


In [12]:
from pprint import pprint

if grid_tuning_job_result.get("BestTrainingJob", None):
    print("Best model found so far:")
    pprint(grid_tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")

Best model found so far:
{'CreationTime': datetime.datetime(2026, 3, 31, 15, 2, 39, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:mse',
                                                 'Value': 1.6895899772644043},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 3, 31, 15, 3, 29, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:training-job/xg-grid-tuningjob-31-14-54-24-015-32ba76a3',
 'TrainingJobName': 'xg-grid-tuningjob-31-14-54-24-015-32ba76a3',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2026, 3, 31, 15, 2, 48, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '1.5',
                          'eta': '0.3',
                          'max_depth': '7',
                          'min_child_weight': '7'}}


In [13]:
grid_tuner = sagemaker.HyperparameterTuningJobAnalytics(grid_tuning_job_name)

full_df = grid_tuner.dataframe()

if len(full_df) > 0:
    df = full_df[full_df["FinalObjectiveValue"] > -float("inf")]
    if len(df) > 0:
        df = df.sort_values("FinalObjectiveValue", ascending=is_minimize)
        print("Number of training jobs with valid objective: %d" % len(df))
        print({"lowest": min(df["FinalObjectiveValue"]), "highest": max(df["FinalObjectiveValue"])})
        pd.set_option("display.max_colwidth", None)  # Don't truncate TrainingJobName
    else:
        print("No training jobs have reported valid results yet.")

full_df

Number of training jobs with valid objective: 16
{'lowest': 1.6895899772644043, 'highest': 2.55964994430542}


,alpha,eta,max_depth,min_child_weight,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,1.5,0.7,7.0,7.0,xg-grid-tuningjob-31-14-54-24-016-0a7c2698,Completed,2.55942,2026-03-31 15:03:11+00:00,2026-03-31 15:03:50+00:00,39.0
1,1.5,0.3,7.0,7.0,xg-grid-tuningjob-31-14-54-24-015-32ba76a3,Completed,1.68959,2026-03-31 15:02:48+00:00,2026-03-31 15:03:29+00:00,41.0
2,1.5,0.7,7.0,3.0,xg-grid-tuningjob-31-14-54-24-014-4e2fb9c6,Completed,2.55942,2026-03-31 15:03:01+00:00,2026-03-31 15:05:11+00:00,130.0
3,1.5,0.3,7.0,3.0,xg-grid-tuningjob-31-14-54-24-013-bc496599,Completed,1.68959,2026-03-31 15:02:12+00:00,2026-03-31 15:02:46+00:00,34.0
4,0.5,0.7,7.0,7.0,xg-grid-tuningjob-31-14-54-24-012-6ff86558,Completed,2.55965,2026-03-31 15:01:52+00:00,2026-03-31 15:02:31+00:00,39.0
5,0.5,0.3,7.0,7.0,xg-grid-tuningjob-31-14-54-24-011-e9754cdb,Completed,1.68976,2026-03-31 15:01:28+00:00,2026-03-31 15:02:07+00:00,39.0
6,0.5,0.7,7.0,3.0,xg-grid-tuningjob-31-14-54-24-010-5f6835fb,Completed,2.55965,2026-03-31 15:00:52+00:00,2026-03-31 15:01:26+00:00,34.0
7,0.5,0.3,7.0,3.0,xg-grid-tuningjob-31-14-54-24-009-581cf41c,Completed,1.68976,2026-03-31 14:59:56+00:00,2026-03-31 15:00:35+00:00,39.0
8,1.5,0.7,3.0,7.0,xg-grid-tuningjob-31-14-54-24-008-a724458c,Completed,2.55942,2026-03-31 14:59:55+00:00,2026-03-31 15:00:34+00:00,39.0
9,1.5,0.3,3.0,7.0,xg-grid-tuningjob-31-14-54-24-007-d88ca324,Completed,1.68959,2026-03-31 14:59:49+00:00,2026-03-31 15:00:23+00:00,34.0


In [18]:
grid_hover = HoverHelper(grid_tuner)

p = figure(width=900, height=400, tools=grid_hover.tools(), x_axis_type="datetime")
p.scatter(source=df, x="TrainingStartTime", y="FinalObjectiveValue",  marker="circle", size=10)
show(p)

In [19]:
ranges = grid_tuner.tuning_ranges
figures = []
for hp_name, hp_range in ranges.items():
    categorical_args = {}
    if hp_range.get("Values"):
        # This is marked as categorical.  Check if all options are actually numbers.
        def is_num(x):
            try:
                float(x)
                return 1
            except:
                return 0

        vals = hp_range["Values"]
        if sum([is_num(x) for x in vals]) == len(vals):
            # Bokeh has issues plotting a "categorical" range that's actually numeric, so plot as numeric
            print("Hyperparameter %s is tuned as categorical, but all values are numeric" % hp_name)
        else:
            # Set up extra options for plotting categoricals.  A bit tricky when they're actually numbers.
            categorical_args["x_range"] = vals

    # Now plot it
    p = figure(
        width=500,
        height=500,
        title="Objective vs %s" % hp_name,
        tools=grid_hover.tools(),
        x_axis_label=hp_name,
        y_axis_label=objective_name,
        **categorical_args,
    )
    p.scatter(source=df, x=hp_name, y="FinalObjectiveValue", marker="circle", size=10)
    figures.append(p)
show(bokeh.layouts.Column(*figures))

Hyperparameter eta is tuned as categorical, but all values are numeric
Hyperparameter min_child_weight is tuned as categorical, but all values are numeric
Hyperparameter alpha is tuned as categorical, but all values are numeric
Hyperparameter max_depth is tuned as categorical, but all values are numeric


## Random Search
Unlike GridSearch, RandomSearch allows continuous and integer parameters. Therefore, we will use the same ranges as the Bayesian algorithm to enforce consistency.

In [20]:
from time import gmtime, strftime, sleep

rand_tuning_job_name = "xg-rand-tuningjob-" + strftime("%d-%H-%M-%S", gmtime())

print(rand_tuning_job_name)

rand_tuning_job_config = {
    "ParameterRanges": {
        "CategoricalParameterRanges": [],
        "ContinuousParameterRanges": [
            {
                "MaxValue": "1",
                "MinValue": "0",
                "Name": "eta",
            },
            {
                "MaxValue": "10",
                "MinValue": "1",
                "Name": "min_child_weight",
            },
            {
                "MaxValue": "2",
                "MinValue": "0",
                "Name": "alpha",
            },
        ],
        "IntegerParameterRanges": [
            {
                "MaxValue": "10",
                "MinValue": "1",
                "Name": "max_depth",
            }
        ],
    },
    "ResourceLimits": {"MaxNumberOfTrainingJobs": 10, "MaxParallelTrainingJobs": 3},
    "Strategy": "Random",
    "HyperParameterTuningJobObjective": {"MetricName": "validation:mse", "Type": "Minimize"},
}

xg-rand-tuningjob-02-21-33-37


In [58]:
# Initiate training jobs for random strategy
smclient.create_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=rand_tuning_job_name,
    HyperParameterTuningJobConfig=rand_tuning_job_config,
    TrainingJobDefinition=training_job_definition,
)

{'HyperParameterTuningJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:hyper-parameter-tuning-job/xg-rand-tuningjob-31-15-11-25',
 'ResponseMetadata': {'RequestId': '83eb6e14-d81c-46a9-bd66-d4e8576cd832',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '83eb6e14-d81c-46a9-bd66-d4e8576cd832',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '130',
   'date': 'Tue, 31 Mar 2026 15:11:28 GMT'},
  'RetryAttempts': 0}}

In [22]:
smclient.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=rand_tuning_job_name)[
    "HyperParameterTuningJobStatus"
]

'Completed'

In [23]:
# run this cell to check current status of hyperparameter tuning job
rand_tuning_job_result = smclient.describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=rand_tuning_job_name
)

status = rand_tuning_job_result["HyperParameterTuningJobStatus"]
if status != "Completed":
    print("Reminder: the tuning job has not been completed.")

job_count = rand_tuning_job_result["TrainingJobStatusCounters"]["Completed"]
print("%d training jobs have completed" % job_count)

objective = rand_tuning_job_result["HyperParameterTuningJobConfig"]["HyperParameterTuningJobObjective"]
is_minimize = objective["Type"] != "Maximize"
objective_name = objective["MetricName"]

10 training jobs have completed


In [24]:
from pprint import pprint

if rand_tuning_job_result.get("BestTrainingJob", None):
    print("Best model found so far:")
    pprint(rand_tuning_job_result["BestTrainingJob"])
else:
    print("No training jobs have reported results yet.")

Best model found so far:
{'CreationTime': datetime.datetime(2026, 3, 31, 15, 11, 39, tzinfo=tzlocal()),
 'FinalHyperParameterTuningJobObjectiveMetric': {'MetricName': 'validation:mse',
                                                 'Value': 1.1409900188446045},
 'ObjectiveStatus': 'Succeeded',
 'TrainingEndTime': datetime.datetime(2026, 3, 31, 15, 14, 17, tzinfo=tzlocal()),
 'TrainingJobArn': 'arn:aws:sagemaker:us-east-1:152554489613:training-job/xg-rand-tuningjob-31-15-11-25-003-7b8ded9d',
 'TrainingJobName': 'xg-rand-tuningjob-31-15-11-25-003-7b8ded9d',
 'TrainingJobStatus': 'Completed',
 'TrainingStartTime': datetime.datetime(2026, 3, 31, 15, 12, 17, tzinfo=tzlocal()),
 'TunedHyperParameters': {'alpha': '0.4310144511087466',
                          'eta': '0.2613746809168078',
                          'max_depth': '10',
                          'min_child_weight': '9.67161259046046'}}


In [25]:
import pandas as pd

rand_tuner = sagemaker.HyperparameterTuningJobAnalytics(rand_tuning_job_name)

full_df = rand_tuner.dataframe()

if len(full_df) > 0:
    df = full_df[full_df["FinalObjectiveValue"] > -float("inf")]
    if len(df) > 0:
        df = df.sort_values("FinalObjectiveValue", ascending=is_minimize)
        print("Number of training jobs with valid objective: %d" % len(df))
        print({"lowest": min(df["FinalObjectiveValue"]), "highest": max(df["FinalObjectiveValue"])})
        pd.set_option("display.max_colwidth", None)  # Don't truncate TrainingJobName
    else:
        print("No training jobs have reported valid results yet.")

full_df

Number of training jobs with valid objective: 10
{'lowest': 1.1409900188446045, 'highest': 2.3621699810028076}


,alpha,eta,max_depth,min_child_weight,TrainingJobName,TrainingJobStatus,FinalObjectiveValue,TrainingStartTime,TrainingEndTime,TrainingElapsedTimeSeconds
0,1.208520,0.679030,8.0,4.197374,xg-rand-tuningjob-31-15-11-25-010-641c0459,Completed,2.36217,2026-03-31 15:16:35+00:00,2026-03-31 15:17:09+00:00,34.0
1,0.459681,0.551424,5.0,7.332392,xg-rand-tuningjob-31-15-11-25-009-d237196b,Completed,1.33215,2026-03-31 15:15:34+00:00,2026-03-31 15:16:14+00:00,40.0
2,1.739148,0.588327,4.0,6.065776,xg-rand-tuningjob-31-15-11-25-008-d3037557,Completed,1.59975,2026-03-31 15:16:31+00:00,2026-03-31 15:17:05+00:00,34.0
3,1.512981,0.934672,7.0,4.807477,xg-rand-tuningjob-31-15-11-25-007-a9fc2fef,Completed,1.87465,2026-03-31 15:15:42+00:00,2026-03-31 15:16:11+00:00,29.0
4,0.990235,0.977093,1.0,8.293159,xg-rand-tuningjob-31-15-11-25-006-04b4798c,Completed,2.11427,2026-03-31 15:14:36+00:00,2026-03-31 15:15:15+00:00,39.0
5,0.037546,0.646826,7.0,1.437707,xg-rand-tuningjob-31-15-11-25-005-c9b6b863,Completed,2.07478,2026-03-31 15:14:34+00:00,2026-03-31 15:15:13+00:00,39.0
6,0.101197,0.895326,10.0,5.232283,xg-rand-tuningjob-31-15-11-25-004-67cce19b,Completed,1.66561,2026-03-31 15:14:35+00:00,2026-03-31 15:15:14+00:00,39.0
7,0.431014,0.261375,10.0,9.671613,xg-rand-tuningjob-31-15-11-25-003-7b8ded9d,Completed,1.14099,2026-03-31 15:12:17+00:00,2026-03-31 15:14:17+00:00,120.0
8,0.292224,0.529400,4.0,9.034181,xg-rand-tuningjob-31-15-11-25-002-f9b28e8e,Completed,1.18403,2026-03-31 15:12:18+00:00,2026-03-31 15:14:17+00:00,119.0
9,1.640093,0.553826,2.0,7.094890,xg-rand-tuningjob-31-15-11-25-001-1e72a66c,Completed,1.34868,2026-03-31 15:12:15+00:00,2026-03-31 15:14:20+00:00,125.0


In [26]:
rand_hover = HoverHelper(rand_tuner)

p = figure(width=900, height=400, tools=rand_hover.tools(), x_axis_type="datetime")
p.scatter(source=df, x="TrainingStartTime", y="FinalObjectiveValue",  marker="circle", size=10)
show(p)

In [27]:
ranges = rand_tuner.tuning_ranges
figures = []
for hp_name, hp_range in ranges.items():
    categorical_args = {}
    if hp_range.get("Values"):
        # This is marked as categorical.  Check if all options are actually numbers.
        def is_num(x):
            try:
                float(x)
                return 1
            except:
                return 0

        vals = hp_range["Values"]
        if sum([is_num(x) for x in vals]) == len(vals):
            # Bokeh has issues plotting a "categorical" range that's actually numeric, so plot as numeric
            print("Hyperparameter %s is tuned as categorical, but all values are numeric" % hp_name)
        else:
            # Set up extra options for plotting categoricals.  A bit tricky when they're actually numbers.
            categorical_args["x_range"] = vals

    # Now plot it
    p = figure(
        width=500,
        height=500,
        title="Objective vs %s" % hp_name,
        tools=rand_hover.tools(),
        x_axis_label=hp_name,
        y_axis_label=objective_name,
        **categorical_args,
    )
    p.scatter(source=df, x=hp_name, y="FinalObjectiveValue", marker="circle", size=10)
    figures.append(p)
show(bokeh.layouts.Column(*figures))

# Analyzing and comparing the results to those obtained with Bayesian Search.

For all three hypertuning algorithms, the metric name "'validation:mse'" was chosen so that they can be compared to one another. Since we are measuring mse, the model with the lowest value indicates the one with best performance (lowest error)

- Bayesian: The best model had the following values: FinalObjectiveValue: 1.07655; alpha: 0.417563; eta: 0.515321; max_depth: 2.0; min_child_weight: 1.394242
- GridSearch: The best model had the following values: FinalObjectiveValue: 1.68959; alpha: 1.5; eta: 0.3; max_depth: 7.0; min_child_weight: 7.0
- RandomSearch: The best model had the following values: FinalObjectiveValue: 1.14099; alpha: 0.431014; eta: 0.261375; max_depth: 10.0; min_child_weight: 9.671613

### Bayesian Vs GridSearch
From the results, we can see that the Bayesian approach identified hyperparameters that allowed its best model to achieve a lower MSE value (1.07655) compared to the best model in the GridSearch algorithm (1.68959). From what I understand, this difference is mainly because the GridSearch approach is restricted to categorical hyperparameters; therefore, even though the four identified hyperparameters (alpha, eta, max_depth, min_child_weight) are continuous or integer parameters, we could only select a small subset of these values to proxy them as categorical values in GridSearch. Since GridSearch is limited by its options, none of the selected hyperparameter combinations yielded a model that came close to the Bayesian approach.

### Bayesian Vs RandomSearch
From the results, we can see that the Bayesian approach identified hyperparameters that allowed its best model to achieve a lower MSE value (1.07655) compared to the best model in the RandomSearch algorithm (1.14099). That being said, the model identified by the RandomSearch algorithm was still fairly close; unlike GridSearch, RandomSearch also allows continuous and integer parameters, which likely contributed to the closer performance. What's interesting is that although the alpha value is similar, the max_depth and min_child_weight parameters were on opposite ends of the range for that value.

# Analyzing the correlation between the objective metric and hyperparameters

### Bayesian
- Max_depth: Some variation, but generally the value of 2.0, 9, and 10 had lower mse values.
- eta: A fair bit of variation, but values around 0.1, 0.5, and 0.7 yielded lower mse values.
- min_child_weight: Generally, values below 3 yielded lower mse values
- alpha: generally, values below 0.5 yielded lower mse values.

### GridSearch
- max_depth: The values 3 and 7 both correlated with a higher and lower mse value.
- eta: The value 0.3 reliable correlated with a lower mse value than 0.7
- min_child_weight: The values 3 and 7 both correlated with a higher or lower mse value.
- alpha: The values 0.5 and 1.5 both correlated with a higher or lower mse value.

# RandomSearch
- max_depth: There is some variation, but the values between 2-5 and the value 10 generally correlated with lower mse values.
- eta: There is some variation, but values below 0.6 generally correlated with a lower mse value.
- min_child_weight: There is some variation, but generally the values greater than 6 (with one exception at 8.293) correlated with lower mse values
- alpha: There is some variation across the range; from the plots, values between 0.29 and 0.5 and the value 1.640 correlated with lower mse values.

# Develop hypotheses regarding potential hyperparameter values for improved model performance.
Based on a review of all three values, the following are worth investigating:
- max_depth: There is some variation, but a value of 10 might lead to a lower mse.
- eta: A value of close to 0.5 might lead to a lower mse.
- min_child_weight: The correlations for this value seems algorithm-dependent; will choose smaller value since Bayesian algorithm had lowest mse.
- alpha: A value close to 0.4 or so might lead to a lower mse.

# Start a new training job with selected hyperparameters based on the hypotheses and validate the results.

In [41]:
from sagemaker import image_uris
region = boto3.Session().region_name
instance_type = 'ml.m5.xlarge'
xg_version = '1.7-1'
train_channel = "train"
val_channel = "validation"
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "train/train.csv")
).upload_file("train.csv")
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "validation/validation.csv")
).upload_file("validation.csv")

In [42]:
# Select XGBoost algorithm for new training job (not tuning job)
from sagemaker import image_uris
xg_container = image_uris.retrieve(
    region=region,
    framework='xgboost',
    version=xg_version,
    instance_type=instance_type,
    image_scope='training'
)

In [49]:
import time
xg_job = f"XGBoost-{time.strftime('%Y-%m-%d-%H-%M-%S', time.gmtime())}"
print(f"XGBoost Job Name is: {xg_job}")

xg_training_params = {
    "RoleArn": role,
    "TrainingJobName": xg_job,
    "AlgorithmSpecification": {"TrainingImage": xg_container, "TrainingInputMode": "File"},
    "ResourceConfig": {"InstanceCount": 1, "InstanceType": instance_type, "VolumeSizeInGB": 30},
    "InputDataConfig": [
        {
            "ChannelName": 'train',
            "ContentType": 'text/csv',
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": "s3://{}/{}/{}/".format(bucket, prefix, train_channel),
                    "S3DataDistributionType": "ShardedByS3Key",
                }
            },
            "CompressionType": "None",
            "RecordWrapperType": "None",
        },
        {
            "ChannelName": 'validation',
            "ContentType": 'text/csv',
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": "s3://{}/{}/{}/".format(bucket, prefix, val_channel),
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "CompressionType": "None",
            "RecordWrapperType": "None",
        },
    ],
    "OutputDataConfig": {"S3OutputPath": "s3://{}/{}/".format(bucket, prefix)},
    "HyperParameters": {
        "objective": "reg:squarederror",  # regression
        "num_round": "30",
        "eval_metric": 'mse',
        "max_depth": "10",               # Controls model complexity
        "eta": "0.5",                    # Learning rate
        "min_child_weight": "0.1394",
        "alpha": "0.42"
    },
    "StoppingCondition": {"MaxRuntimeInSeconds": 60 * 60},
}

XGBoost Job Name is: XGBoost-2026-04-03-01-56-09


In [50]:
%%time

# Train XG-Boost Job
smclient.create_training_job(**xg_training_params)

status = smclient.describe_training_job(TrainingJobName=xg_job)["TrainingJobStatus"]
print(status)
smclient.get_waiter("training_job_completed_or_stopped").wait(TrainingJobName=xg_job)
if status == "Failed":
    message = sm.describe_training_job(TrainingJobName=xg_job)["FailureReason"]
    print("Training failed with the following error: {}".format(message))
    raise Exception("Training job failed")

InProgress
CPU times: user 60.5 ms, sys: 5.27 ms, total: 65.8 ms
Wall time: 4min


# Hosting the XGBoost model in AWS to make inferences

In [51]:
xg_hosting_container = {
    "Image": xg_container,
    "ModelDataUrl": smclient.describe_training_job(TrainingJobName=xg_job)["ModelArtifacts"][
        "S3ModelArtifacts"
    ],
}

xg_model_response = smclient.create_model(
    ModelName=xg_job, ExecutionRoleArn=role, PrimaryContainer=xg_hosting_container
)

print(xg_model_response["ModelArn"])

arn:aws:sagemaker:us-east-1:152554489613:model/XGBoost-2026-04-03-01-56-09


# Defining model endpoint configurations

In [52]:
xg_endpoint_config = "A8-xg-endpoint-config-" + time.strftime(
    "%Y-%m-%d-%H-%M-%S", time.gmtime()
)
print(xg_endpoint_config)
xg_endpoint_config_response = smclient.create_endpoint_config(
    EndpointConfigName=xg_endpoint_config,
    ProductionVariants=[
        {
            "InstanceType": instance_type,
            "InitialInstanceCount": 1,
            "ModelName": xg_job,
            "VariantName": "AllTraffic",
        }
    ],
)

print("Endpoint Config Arn: " + xg_endpoint_config_response["EndpointConfigArn"])

A8-xg-endpoint-config-2026-04-03-02-01-54
Endpoint Config Arn: arn:aws:sagemaker:us-east-1:152554489613:endpoint-config/A8-xg-endpoint-config-2026-04-03-02-01-54


# Creating the endpoint that connects to the trained model.

In [54]:
%%time

xg_endpoint = "A8-xg-endpoint-" + time.strftime("%Y%m%d%H%M", time.gmtime())
print(xg_endpoint)
xg_endpoint_response = smclient.create_endpoint(
    EndpointName=xg_endpoint, EndpointConfigName=xg_endpoint_config
)
print(xg_endpoint_response["EndpointArn"])

resp = smclient.describe_endpoint(EndpointName=xg_endpoint)
status = resp["EndpointStatus"]
print("Status: " + status)

smclient.get_waiter("endpoint_in_service").wait(EndpointName=xg_endpoint)

resp = smclient.describe_endpoint(EndpointName=xg_endpoint)
status = resp["EndpointStatus"]
print("Arn: " + resp["EndpointArn"])
print("Status: " + status)

if status != "InService":
    raise Exception("Endpoint creation did not succeed")

A7-xg-endpoint-202604030202
arn:aws:sagemaker:us-east-1:152554489613:endpoint/A7-xg-endpoint-202604030202
Status: Creating
Arn: arn:aws:sagemaker:us-east-1:152554489613:endpoint/A7-xg-endpoint-202604030202
Status: InService
CPU times: user 43.5 ms, sys: 110 μs, total: 43.6 ms
Wall time: 3min 31s


In [56]:
def np2csv(arr):
    csv = io.BytesIO() #the function gets an array (Numpy array) and creates an in-memory binary buffer named csv
    np.savetxt(csv, arr, delimiter=",", fmt="%g") # write the array 'arr' to csv object, columns should be seperated by commas. The fmt="%g" removes unneccesary decimals when saving and use scientific notation.
    # In the following line:
    # csv.getvalue() retrieves the entire contents of the buffer csv as a byte string.
    # .decode() converts the byte string into a normal Python string by decoding it using the default UTF-8 encoding.
    #.rstrip() removes any trailing whitespace or newlines from the end of the string.
    return csv.getvalue().decode().rstrip()

In [58]:
train_X = train_data.drop(["y_no", "y_yes"], axis=1)
test_X = test_data.drop(["y_no", "y_yes"], axis=1)
train_y = train_data["y_yes"]
test_y = test_data["y_yes"]

# Make an inference request to the model and store prediction values for evaluation

In [60]:
%%time

import io
runtime = boto3.client("runtime.sagemaker")

payload = np2csv(test_X)

response = runtime.invoke_endpoint(
    EndpointName=xg_endpoint, ContentType="text/csv", Body=payload
)

result = response["Body"].read().decode()

xg_test_pred = np.array([float(x) for x in result.split() if x.strip()])

CPU times: user 71.6 ms, sys: 0 ns, total: 71.6 ms
Wall time: 336 ms


In [61]:
print(xg_test_pred)

[1.45681188e-05 1.45681188e-05 1.45681188e-05 ... 1.45681188e-05
 1.45681188e-05 1.45681188e-05]


In [62]:
test_mae_linear = np.mean(np.abs(test_y - xg_test_pred)) # Mean Absolute Error (MAE) of the predictions vs real target values.
#The following line calculates the MAE baseline model using a very simple strategy for predictions: 
#it always predicts the median value of the target variable from the training dataset, which is np.median(train_y).
#The idea is to provide a simple comparison to see if the linear model is performing better than a model that always guesses the median value.
test_mae_baseline = np.mean(
    np.abs(test_y - np.median(train_y))
)  

print("Test MAE Baseline :", round(test_mae_baseline, 3))
print("Test MAE Linear:", round(test_mae_linear, 3))

Test MAE Baseline : 0.117
Test MAE Linear: 0.117


### Validate the results
Based on the above, we see that the model has a MAE of 0.117, compared to the validation MAE of 1.07655 in the models trained while tuning hyperparameters via Bayesian algorithm.
Therefore, it does seem that selecting hyperparameters based on the hypothesis was successful.